# Mini projekt PAD - scrapper danych o oponach
## Jakub Michalak S20034
### Opis projektu
- Projekt polega na pobraniu danych o oponach z dwóch sklepów internetowych: sklep opon i oponeo
- Dane pobrane z obu sklepów zostaną zapisane w plikach CSV
- Dane zostaną oczyszczone i przygotowane do analizy

In [12]:
import pandas as pd
import re
import time

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException

### Konfiguracja drivera do scrapowania danych 
- **Uwaga**: testowane na szerokości okna 1110 pikseli
- Ustawienie szerokości okna na 1100 pikseli pozwala na poprawne działanie skryptów do scrapowania danych ze stron sklep opon i oponeo. Przy wyższej szerokości okna mogą wystąpić problemy z lokalizacją elementów na stronie (np. oceny).

In [117]:
download_service = Service()
driver = webdriver.Chrome(service=download_service)
driver.set_window_size(1100, 800)

sklep_opon_base_url = "https://www.sklepopon.com/szukaj-opony?sezon=zimowe&rozmiar=205/55R16&ofs="
oponeo_base_url = "https://www.oponeo.pl/wybierz-opony/s=1/zimowe/t=1/osobowe/r=1/205-55-r16"

### Funkcje obsługujące scrapowanie danych ze stron sklep opon i oponeo

In [116]:
def close_sklep_opon_popups(outer_driver):
    try:
        btn_cookie = outer_driver.find_element(By.CSS_SELECTOR, "#klaro > div > div > div > div > div > button")
        btn_cookie.click()
        print("Przycisk akceptacji ciasteczek został kliknięty.")
    except NoSuchElementException:
        print("Przycisk akceptacji ciasteczek nie został znaleziony.")
    except Exception as exception:
        print("Nie udało się kliknąć przycisku akceptacji ciasteczek:", exception)
    
    try:
        outer_driver.execute_script("""
            const shadowHost = document.querySelector("body > div.gr-visual-prompt");
            if (shadowHost) {
                const shadowRoot = shadowHost.shadowRoot;
                const closeButton = shadowRoot.querySelector("div > div:nth-child(2) > button:nth-child(1)");
                if (closeButton) {
                    closeButton.click();
                    console.log("Okienko powiadomień zostało zamknięte.");
                } else {
                    console.log("Nie znaleziono przycisku zamknięcia powiadomień.");
                }
            } else {
                console.log("Okno powiadomień nie zostało znalezione.");
            }
        """)
    except Exception as exception:
        print("Nie udało się zamknąć okienka powiadomień:", exception)

def close_oponeo_popup(outer_driver):
    try:
        reject_button = outer_driver.find_element(By.CSS_SELECTOR, "#consentsBar > div.buttonsContainer.container > div > span.reject")
        reject_button.click()
        print("Okienko prywatności zostało zamknięte.")
    except NoSuchElementException:
        print("Okienko prywatności nie jest widoczne lub zostało już zamknięte.")
    except Exception as e:
        print("Wystąpił błąd podczas zamykania okienka prywatności:", e)
        
def load_sklep_opon_tire_data(outer_driver):
    scrapped_data = []
    try:
        opony_elements = outer_driver.find_elements(By.CSS_SELECTOR, 'div[data-c-name="listing-products-element"]')
            
        class_mapping = {
            "Premium": "Premium",
            "Średnia": "Średnia",
            "Średniej": "Średnia",
            "Ekonomiczna": "Ekonomiczna",
            "Ekonomicznej": "Ekonomiczna"
        }
        
        for opona_element in opony_elements:
            
            try:
                load_index = opona_element.find_element(By.CSS_SELECTOR, 'li[data-attribute-code="li"]').text
                speed_index = opona_element.find_element(By.CSS_SELECTOR, 'li[data-attribute-code="si"]').text
            except NoSuchElementException:
                load_index = None
                speed_index = None
            
            noise_level = None
            try:
                noise_level_elements = opona_element.find_elements(By.CSS_SELECTOR, "span.self-center.tracking-tighter.sm\\:tracking-normal")
                for noise_level_element in noise_level_elements:
                    text = noise_level_element.text
                    match = re.search(r'\d+', text)
                    if match:
                        noise_level = int(match.group())
            except (NoSuchElementException, IndexError):
                pass
            
            etykieta_elements = opona_element.find_elements(By.CSS_SELECTOR, 'span.icon-fuel-new ~ span, span.icon-rain-new ~ span, span.icon-speaker-new ~ span')
            fuel_index = etykieta_elements[0].text if len(etykieta_elements) > 0 else None
            wet_grip_index = etykieta_elements[1].text if len(etykieta_elements) > 1 else None
            noise_index = etykieta_elements[2].text.split(" ")[0] if len(etykieta_elements) > 2 else None
            
            try:
                tire_class_element = opona_element.find_element(By.XPATH, ".//*[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'klas')]")
                tire_class_text = tire_class_element.text.lower().replace("w klasie ", "").replace("klasa ", "").strip().capitalize()
                tire_class = class_mapping.get(tire_class_text, tire_class_text)
            except NoSuchElementException:
                tire_class = None
                
            try:
                user_rating_element = opona_element.find_element(By.XPATH, ".//li[contains(@class, 'xl:hidden')]//span[contains(@class, 'ml-1')]")
                user_rating = float(user_rating_element.text.replace(",", "."))
            except NoSuchElementException:
                user_rating = None
                
            price = float(opona_element.get_attribute('data-ee-product-properties').split(";")[3].split(":")[1])
            
            # Pobranie dostępności
            try:
                availability_element = opona_element.find_element(By.CSS_SELECTOR, "div.tooltip-product-listing")
                availability_code = availability_element.get_attribute("data-attribute-code")
                
                if availability_code == "item_availability_tooltip_high":
                    availability = "full"
                elif availability_code == "item_availability_tooltip_medium":
                    availability = "medium"
                elif availability_code == "item_availability_tooltip_low":
                    availability = "low"
                elif availability_code == "item_availability_tooltip_last":
                    availability = "last"
                else:
                    availability = None
            except NoSuchElementException:
                availability = None
                
            tire_data = {
                "name": opona_element.get_attribute("data-ee-product-properties").split(";")[0].split(":")[1],
                "brand": opona_element.get_attribute("data-ee-product-properties").split(";")[4].split(":")[1],
                "model": opona_element.get_attribute("data-ee-product-properties").split(";")[6].split(":")[1],
                "size": opona_element.get_attribute("data-ee-product-properties").split(";")[5].split(":")[1],
                "load_index": load_index,
                "speed_index": speed_index,
                "fuel_index": fuel_index,
                "wet_grip_index": wet_grip_index,
                "noise_index": noise_index,
                "noise_level": noise_level,
                "class": tire_class,
                "user_rating": user_rating,
                "price": price,
                "availability": availability
            }
            
            scrapped_data.append(tire_data)
    finally:
        pass
    return scrapped_data   

def load_next_oponeo_page(web_driver, current_page_number):
    try:
        next_page_button = web_driver.find_element(By.ID, f"_ctPgrp_pi{current_page_number}i")
        next_page_button.click()
        time.sleep(2)
        return True
    except NoSuchElementException:
        return False
    
def load_oponeo_tire_data(outer_driver):
    scrapped_data = []
    products = outer_driver.find_elements(By.CLASS_NAME, "product")
    
    for product in products:
        try:
            try:
                link_element = product.find_element(By.CSS_SELECTOR, ".productName a")
                nazwa = link_element.get_attribute("title")
            except NoSuchElementException:
                nazwa = product.find_element(By.CLASS_NAME, "productName").text

            noise = product.find_element(By.CSS_SELECTOR, ".icon-noise em").text
            match = re.search(r'\d+', noise)
            if match:
                noise_level = int(match.group())
            else:
                noise_level = int(product.find_element(By.CSS_SELECTOR, ".icon-noise em").text.split()[1].replace("dB", "").strip())
             
            noise_index = product.find_element(By.CSS_SELECTOR, ".icon-noise em").text.split()[0] 
            noise_index_text = noise_index if noise_index in {"A", "B", "C", "D", "E", "F"} and len(noise_index) == 1 else None   
                
            try:
                user_rating = product.find_element(By.CSS_SELECTOR, ".productRating .note").text
                user_rating = user_rating.replace(',', '.')
            except NoSuchElementException:
                user_rating = None
                
            # Pobranie poziomu dostępności
            try:
                stock_level_element = product.find_element(By.CSS_SELECTOR, ".stockLevel")
                stock_level_class = stock_level_element.get_attribute("class").split()[-1]
                
                if stock_level_class == "full":
                    availability = "full"
                elif stock_level_class == "medium":
                    availability = "medium"
                elif stock_level_class == "low":
                    availability = "low"
                else:
                    availability = None
            except NoSuchElementException:
                availability = None
                
            tire_info = {
                "name": nazwa,
                "brand": product.find_element(By.CLASS_NAME, "producerName").text,
                "model": product.find_element(By.CLASS_NAME, "modelName").text,
                "size": product.find_element(By.CLASS_NAME, "modelSize").text,
                "load_index": product.find_element(By.XPATH, ".//span[@data-tp='TireLoadIndex']/em").text,
                "speed_index": product.find_element(By.XPATH, ".//span[@data-tp='TireSpeedIndex']/em").text,
                "fuel_index": product.find_element(By.CSS_SELECTOR, ".icon-fuel em").text,
                "wet_grip_index": product.find_element(By.CSS_SELECTOR, ".icon-rain em").text,
                "noise_index": noise_index_text,
                "noise_level": noise_level,
                "class": product.find_element(By.CLASS_NAME, "class").text.replace("KLASA ", "").capitalize(),
                "user_rating": user_rating,
                "price": product.find_element(By.CLASS_NAME, "priceValue").text,
                "availability": availability
            }
            
            scrapped_data.append(tire_info)

        except NoSuchElementException:
            pass
            
    return scrapped_data    

### Kod do pobierania danych ze strony sklep opon

In [118]:
sklep_opon_tires_data = []
offset = 0

while True:
    url = f"{sklep_opon_base_url}{offset}"
    driver.get(url)
    time.sleep(4)
    
    if offset == 0:
        close_sklep_opon_popups(driver)
    
    tires_data = load_sklep_opon_tire_data(driver)
    sklep_opon_tires_data.extend(tires_data)
    
    offset += 20
    if len(driver.find_elements(By.CSS_SELECTOR, 'div[data-c-name="listing-products-element"]')) == 0:
        print("Brak nowych danych. Koniec paginacji.")
        break

df_sklep_opon = pd.DataFrame(sklep_opon_tires_data)
display(df_sklep_opon)

Przycisk akceptacji ciasteczek został kliknięty.
Brak nowych danych. Koniec paginacji.


,name,brand,model,size,load_index,speed_index,fuel_index,wet_grip_index,noise_index,noise_level,class,user_rating,price,availability
0,Wintrac 205/55 R16 91 H,Vredestein,Wintrac,205/55 R16,91,H,C,B,B,70.0,Premium,5.4,376.99,full
1,Winguard Snow'G WH2 205/55 R16 91 H,Nexen,Winguard Snow'G WH2,205/55 R16,91,H,D,C,B,70.0,Średnia,5.3,310.00,full
2,DIMAX ALPINE 205/55 R16 94 H,Radar,DIMAX ALPINE,205/55 R16,94,H,D,C,A,69.0,Ekonomiczna,5.2,225.49,full
3,Frigo HP2 205/55 R16 91 H,Dębica,Frigo HP2,205/55 R16,91,H,C,C,B,72.0,Ekonomiczna,5.2,299.00,full
4,Frigo 2 205/55 R16 91 T,Dębica,Frigo 2,205/55 R16,91,T,C,C,B,71.0,Ekonomiczna,5.1,239.00,full
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401,Winter Sottozero Serie II 205/55 R16 94 V,Pirelli,Winter Sottozero Serie II,205/55 R16,94,V,C,C,B,72.0,Premium,5.5,1059.57,last
402,WINTERPRO2 (EVO)* 205/55 R16 91 T,Gt radial,WINTERPRO2 (EVO)*,205/55 R16,91,T,D,B,B,70.0,None,NaN,340.47,last
403,Ultra Grip 8 205/55 R16 91 T,Goodyear,Ultra Grip 8,205/55 R16,91,T,D,D,B,71.0,Premium,5.2,374.43,last
404,Blizzak LM005 205/55 R16 94 V,Bridgestone,Blizzak LM005,205/55 R16,94,V,C,A,B,71.0,Premium,5.5,643.11,last


### Kod do pobierania danych ze strony oponeo

In [111]:
driver.get(oponeo_base_url)
close_oponeo_popup(driver)

all_tires_data = []

page_number = 1
while True:
    tires_data = load_oponeo_tire_data(driver)
    all_tires_data.extend(tires_data)
    
    page_number += 1
    if not load_next_oponeo_page(driver, page_number):
        print("Brak nowych danych. Koniec paginacji.")
        break

df_oponeo = pd.DataFrame(all_tires_data)
display(df_oponeo)

Okienko prywatności zostało zamknięte.
Brak nowych danych. Koniec paginacji.


,name,brand,model,size,load_index,speed_index,fuel_index,wet_grip_index,noise_index,noise_level,class,user_rating,price,availability
0,Bridgestone Blizzak LM005 205/55 R16 91 H,Bridgestone,Blizzak LM005,205/55 R16,91,H,C,A,B,71,Premium,4.7,459,full
1,Michelin Alpin 7 205/55 R16 91 H,Michelin,Alpin 7,205/55 R16,91,H,C,B,B,71,Premium,4.8,467,full
2,Dębica Frigo 2 205/55 R16 91 T,Dębica,Frigo 2,205/55 R16,91,T,C,C,B,71,Ekonomiczna,4.2,252,full
3,Kormoran Snow 205/55 R16 91 H,Kormoran,Snow,205/55 R16,91,H,D,C,B,72,Ekonomiczna,4.5,249,full
4,Firemax FM805+ 205/55 R16 91 H,Firemax,FM805+,205/55 R16,91,H,D,C,A,67,Ekonomiczna,4.4,209,full
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
217,Continental ContiWinterContact TS830 P 205/55 ...,Continental,ContiWinterContact TS830 P,205/55 R16,91,H,D,C,B,72,Premium,4.4,760,medium
218,Goodyear UG Performance 2 205/55 R16 91 H RUN ...,Goodyear,UG Performance 2,205/55 R16,91,H,D,C,B,72,Premium,4.3,778,full
219,Bridgestone Blizzak LM005 205/55 R16 94 V XL,Bridgestone,Blizzak LM005,205/55 R16,94,V,C,A,B,71,Premium,4.7,854,medium
220,Pirelli SottoZero Serie 3 205/55 R16 91 H RUN ...,Pirelli,SottoZero Serie 3,205/55 R16,91,H,D,B,B,72,Premium,4.6,981,full


### Zamknięcie drivera po scrapowaniu danych

In [121]:
driver.quit()

### Zapisanie danych do plików CSV
- służy to do zapisania danych do plików CSV, aby można było je łatwo odtworzyć w przyszłości
- dane zapisane w plikach CSV można wczytać do DataFrame za pomocą funkcji `pd.read_csv`

In [120]:
# df_sklep_opon.to_csv("data/miniprojekt/sklep_opon.csv", index=False)
# df_oponeo.to_csv("data/miniprojekt/oponeo.csv", index=False)
df_sklep_opon = pd.read_csv("data/miniprojekt/sklep_opon.csv", delimiter=",")
df_oponeo = pd.read_csv("data/miniprojekt/oponeo.csv", delimiter=",")

### Weryfikacja poprawnych wartości w każdej kolumnie

In [122]:
print("sklep opon")
print("typy danych w kolumnach")
print(df_sklep_opon.dtypes)
print("unikalne wartości w kolumnach")
for column in df_sklep_opon.columns:
    if column not in ["name", "brand", "model"]:
        print(column, df_sklep_opon[column].unique())
print("oponeo")
print("typy danych w kolumnach")
print(df_oponeo.dtypes)
print("unikalne wartości w kolumnach")
for column in df_oponeo.columns:
    if column not in ["name", "brand", "model"]:
        print(column, df_oponeo[column].unique())

sklep opon
typy danych w kolumnach
name               object
brand              object
model              object
size               object
load_index         object
speed_index        object
fuel_index         object
wet_grip_index     object
noise_index        object
noise_level       float64
class              object
user_rating       float64
price             float64
availability       object
dtype: object
unikalne wartości w kolumnach
size ['205/55 R16']
load_index ['91' '94']
speed_index ['H' 'T' 'V' 'R']
fuel_index ['C' 'D' 'B' None 'E']
wet_grip_index ['B' 'C' 'D' 'A' None 'E']
noise_index ['B' 'A' None 'C']
noise_level [70. 69. 72. 71. nan 67. 68.]
class ['Premium' 'Średnia' 'Ekonomiczna' None]
user_rating [5.4 5.3 5.2 5.1 5.6 5.  5.5 nan 4.3 4.4 6.  1.  4.8 4.6 4.7 2.3 4.9 4.5
 5.7 4.2 2.5 1.4]
price [ 376.99  310.    225.49  299.    239.    322.    335.    244.99  227.
  385.    320.    460.    289.    224.99  399.    360.    343.99  345.
  329.13  402.99  450.99  370.    435

# Czyszczenie i przygotowanie danych

### #0. konwersja kolumn do typów numerycznych

In [123]:
# Konwersja kolumn do typów numerycznych w zbiorze df_oponeo
df_oponeo['noise_level'] = pd.to_numeric(df_oponeo['noise_level'], errors='coerce').astype('Int64')
df_oponeo['user_rating'] = pd.to_numeric(df_oponeo['user_rating'], errors='coerce').astype(float)
df_oponeo['price'] = pd.to_numeric(df_oponeo['price'], errors='coerce').astype(float)

# Konwersja kolumn do typów numerycznych w zbiorze df_sklep_opon
df_sklep_opon['noise_level'] = pd.to_numeric(df_sklep_opon['noise_level'], errors='coerce').astype('Int64')
df_sklep_opon['user_rating'] = pd.to_numeric(df_sklep_opon['user_rating'], errors='coerce').astype(float)
df_sklep_opon['price'] = pd.to_numeric(df_sklep_opon['price'], errors='coerce').astype(float)

### Funkcje pomocnicze

In [124]:
def find_closest_key(value, param_map):
        closest_key = min(param_map, key=lambda k: abs(param_map[k] - value))
        return closest_key

def find_closest_value(value, param_map):
        closest_key = min(param_map, key=lambda k: abs(param_map[k] - value))
        return param_map[closest_key]

### #1. Uzupełnianie noise_index na podstawie noise_level

In [125]:
def update_noise_index_column(df):
    grouped_noise_index = df.groupby('noise_index')['noise_level'].mean().round().astype(int)
    
    if 'C' not in grouped_noise_index:
        if 'B' in grouped_noise_index and 'A' in grouped_noise_index:
            grouped_noise_index['C'] = grouped_noise_index['B'] + (grouped_noise_index['B'] - grouped_noise_index['A'])
    
    noise_map = grouped_noise_index.to_dict()
    
    def apply_update_noise_index(row):
        if pd.notna(row['noise_level']) and pd.isna(row['noise_index']):
            return find_closest_key(row['noise_level'], noise_map)
        return row['noise_index']
    
    df['noise_index'] = df.apply(apply_update_noise_index, axis=1)
    
    return df

df_oponeo = update_noise_index_column(df_oponeo)
df_sklep_opon = update_noise_index_column(df_sklep_opon)

print("Zaktualizowano noise_index")
print(df_oponeo['noise_index'].unique())
print(df_sklep_opon['noise_index'].unique())

Zaktualizowano noise_index
['B' 'A' 'C']
['B' 'A' None 'C']


### #2. Uzupełnianie ceny na podstawie średniej ceny i klasy na podstawie ceny

In [127]:
def update_price_column(df):
    class_price_map = df.groupby('class')['price'].mean().to_dict()
    
    def apply_update_price(row):
        if pd.isna(row['price']):
            return find_closest_value(row['price'], class_price_map)
        return row['price']
    
    def apply_update_class(row):
        if pd.isna(row['class']):
            return find_closest_key(row['price'], class_price_map)
        return row['class']
    
    df['price'] = df.apply(apply_update_price, axis=1)
    df['class'] = df.apply(apply_update_class, axis=1)
    
    return df

df_oponeo = update_price_column(df_oponeo)
df_sklep_opon = update_price_column(df_sklep_opon)

print("Zaktualizowano cenę i klasę")
print("oponeo")
print(df_oponeo['price'].unique())
print(df_oponeo['class'].unique())
print("sklep opon")
print(df_sklep_opon['price'].unique())
print(df_sklep_opon['class'].unique())

Zaktualizowano cenę i klasę
oponeo
[459.         467.         252.         249.         209.
 191.         207.         215.         216.         217.
 218.         220.         229.         254.         257.
 264.         269.         293.         297.         299.
 307.         332.         337.         338.         339.
 340.         342.         346.         348.         353.
 357.         373.         379.         382.         383.
 387.         388.         390.         392.         393.
 407.         409.         424.         432.         447.
 452.         466.         468.         474.         477.
 520.         555.         597.         258.         267.
 270.         271.         273.         274.         275.
 279.         283.         287.         292.         294.
 298.         300.         301.         304.         305.
 311.         315.         317.         320.         325.
 330.         331.         333.         334.         335.
 336.         345.         350.      